# Assignment 6: Grounded Language with CLIP and Pragmatic Reference Games

**EECS 183/283A · Due Fri Dec 4, 11:59 PM PT**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zinengtang/cs183-fall26-a6/blob/main/HW6.ipynb)

You will implement CLIP's contrastive loss and zero-shot classifier (Part 1), implement the Rational Speech Acts model (Part 2), and use CLIP as the listener in a reference game (Part 3). There is no training and no GPU: a free Colab CPU runtime is enough.

| Part | Topic | 183 | 283A |
|---|---|---|---|
| 1 | CLIP | 35 | 35 |
| 2 | RSA | 30 | 30 |
| 3 | CLIP + RSA | 35 | 35 |
| 4 | Unknown partners | — | 20 |

**Rules**
- Write code only at `# YOUR CODE HERE`.
- `# GRADED:` cells are autograded, each one on its own. Keep the header line and the function names, put imports inside the cell, and don't use variables from other cells.
- Answer the ★ questions in the **Your answer** cells. They are graded from your PDF.
- **Submit** `HW6.ipynb` and `HW6.pdf` (*File → Print → Save as PDF*) to Gradescope **A6**.

Collaborators: ______

## Setup

In [ ]:
# Colab has everything except possibly `datasets`. Locally: pip install -r requirements.txt
import importlib.util, os, subprocess, sys, urllib.request
if importlib.util.find_spec("datasets") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)

import math, time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw
from IPython.display import display

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)
print("device:", device, "| torch", torch.__version__)

---
# Part 1: CLIP (35 pts)

For a batch of $N$ (image, caption) pairs with unit-normalized embeddings $\mathbf{x}_i$ and $\mathbf{y}_j$, CLIP's logits are $\ell_{ij} = s\,\mathbf{x}_i^\top \mathbf{y}_j$, where $s$ is a learned **logit scale**. The loss averages cross-entropy in both directions: each image picks its caption, and each caption picks its image.

$$\mathcal{L} = \frac{1}{2}\left[\frac{1}{N}\sum_{i} -\log \frac{e^{\ell_{ii}}}{\sum_{j} e^{\ell_{ij}}} \;+\; \frac{1}{N}\sum_{j} -\log \frac{e^{\ell_{jj}}}{\sum_{i} e^{\ell_{ij}}}\right]$$

## 1.1 Contrastive loss (8 pts, autograded)

Implement `clip_contrastive_loss` using `F.cross_entropy`. Normalize the inputs yourself. `logit_scale` is $s$ itself (for example `100.0`), not $\log s$.

In [ ]:
# GRADED: clip_contrastive_loss
import torch
import torch.nn.functional as F


def clip_contrastive_loss(image_emb, text_emb, logit_scale):
    """Symmetric CLIP / InfoNCE loss.

    Args:
        image_emb:   float tensor [N, D], row i is the i-th image embedding (not necessarily unit norm)
        text_emb:    float tensor [N, D], row i is the caption paired with image i
        logit_scale: python float or 0-dim tensor s (the multiplier, NOT log s)
    Returns:
        0-dim tensor: the average of the image->text and text->image cross-entropy losses.
    """
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# Sanity check (not graded): random pairs at s=1 give about log N; aligned pairs at s=100 give about 0.
torch.manual_seed(0)
N, D = 16, 32
a = torch.randn(N, D)
print("random, s=1   :", clip_contrastive_loss(a, torch.randn(N, D), 1.0).item(), " log N =", math.log(N))
print("aligned, s=100:", clip_contrastive_loss(a, 3 * a, 100.0).item())

## Load CLIP and CIFAR-10

We use OpenAI's ViT-B/32 CLIP, the only model you download (about 600 MB), and 1,000 CIFAR-10 test images.

In [ ]:
from transformers import CLIPModel, CLIPProcessor


def load_clip(name):
    model = CLIPModel.from_pretrained(name).to(device).eval()
    processor = CLIPProcessor.from_pretrained(name)
    return model, processor


def _as_tensor(out):
    return out if torch.is_tensor(out) else out.pooler_output


@torch.no_grad()
def encode_images(model, processor, images, batch_size=100):
    feats = []
    for i in range(0, len(images), batch_size):
        x = processor(images=images[i:i + batch_size], return_tensors="pt").to(device)
        feats.append(_as_tensor(model.get_image_features(**x)).float().cpu())
    return torch.cat(feats)


@torch.no_grad()
def encode_texts(model, processor, texts, batch_size=256):
    feats = []
    for i in range(0, len(texts), batch_size):
        x = processor(text=list(texts[i:i + batch_size]), return_tensors="pt", padding=True).to(device)
        feats.append(_as_tensor(model.get_text_features(**x)).float().cpu())
    return torch.cat(feats)


clip_b32, proc_b32 = load_clip("openai/clip-vit-base-patch32")
LOGIT_SCALE = clip_b32.logit_scale.exp().item()
print("learned logit scale s =", round(LOGIT_SCALE, 2))

In [ ]:
from datasets import load_dataset

cifar = load_dataset("uoft-cs/cifar10", split="test",                 # test split only (24 MB)
                     data_files={"test": "plain_text/test-00000-of-00001.parquet"})
CLASSES = cifar.features["label"].names
subset = np.random.RandomState(0).choice(len(cifar), 1000, replace=False)
cifar_images = [cifar[int(i)]["img"].convert("RGB") for i in subset]
cifar_labels = np.array([cifar[int(i)]["label"] for i in subset])
print(CLASSES)

t0 = time.time()
cifar_img_emb = encode_images(clip_b32, proc_b32, cifar_images)      # [1000, 512], NOT normalized
print("encoded", tuple(cifar_img_emb.shape), f"in {time.time() - t0:.1f}s")

fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
for ax, img, y in zip(axes, cifar_images, cifar_labels):
    ax.imshow(img); ax.set_title(CLASSES[y], fontsize=9); ax.axis("off")
plt.show()

### ★ Q1.1 In-batch negatives (4 pts)

The cell below computes your loss on two batches. Batch A has one image per class. Batch B has 5 airplanes and 5 ships. Every caption is `"a photo of a {class}."`.

Report both losses. Why is batch B's loss much higher, even though CLIP classifies these images well? What does the loss assume about the other items in a batch, and when does that assumption fail for web (image, alt-text) data?

In [ ]:
caption = lambda y: f"a photo of a {CLASSES[y]}."
batch_a = [int(np.where(cifar_labels == k)[0][0]) for k in range(10)]
batch_b = list(np.where(cifar_labels == 0)[0][:5]) + list(np.where(cifar_labels == 8)[0][:5])
for name, b in [("A (10 distinct classes)", batch_a), ("B (5 airplanes + 5 ships)", batch_b)]:
    txt = encode_texts(clip_b32, proc_b32, [caption(cifar_labels[i]) for i in b])
    print(f"batch {name}: loss = {clip_contrastive_loss(cifar_img_emb[b], txt, LOGIT_SCALE).item():.3f}")

**Your answer (Q1.1):**

*Write your answer here.*

## 1.2 Zero-shot classification (9 pts, autograded)

Implement two functions:

- `build_zero_shot_weights(class_names, templates, encode_text_fn)` → `[K, D]`. For each class, encode all of its filled-in templates in one call, **normalize each embedding**, average them, and re-normalize.
- `zero_shot_logits(image_emb, class_weights, logit_scale)` → `[N, K]` logits, equal to $s \cdot \cos(\text{image}, \text{class})$.

In [ ]:
# GRADED: zero_shot
import torch
import torch.nn.functional as F


def build_zero_shot_weights(class_names, templates, encode_text_fn):
    """
    Args:
        class_names:    list of K strings, e.g. ["airplane", "automobile", ...]
        templates:      list of T format strings containing "{}", e.g. ["a photo of a {}."]
        encode_text_fn: function list[str] -> float tensor [n, D] (unnormalized)
    Returns:
        float tensor [K, D]; row k is the unit-norm (prompt-ensembled) embedding of class k.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def zero_shot_logits(image_emb, class_weights, logit_scale):
    """
    Args:
        image_emb:     float tensor [N, D] (unnormalized)
        class_weights: float tensor [K, D] (unit norm rows, from build_zero_shot_weights)
        logit_scale:   float s
    Returns:
        float tensor [N, K] of logits s * cos(image_n, class_k).
    """
    # YOUR CODE HERE
    raise NotImplementedError

We compare three prompt settings: the bare class name, one template, and 8 templates from the CLIP paper.

In [ ]:
TEMPLATE_SETS = {
    "bare name": ["{}"],
    "a photo of a {}.": ["a photo of a {}."],
    "8-template ensemble": [
        "a photo of a {}.", "a blurry photo of a {}.", "a low resolution photo of a {}.",
        "a photo of the small {}.", "a photo of the big {}.", "a pixelated photo of a {}.",
        "a cropped photo of a {}.", "a good photo of a {}.",
    ],
}
encode_text_b32 = lambda texts: encode_texts(clip_b32, proc_b32, texts)

zs_results = {}
for name, templates in TEMPLATE_SETS.items():
    W = build_zero_shot_weights(CLASSES, templates, encode_text_b32)
    logits = zero_shot_logits(cifar_img_emb, W, LOGIT_SCALE)
    zs_results[name] = (logits, (logits.argmax(1).numpy() == cifar_labels).mean())
    print(f"{name:>22s}: zero-shot accuracy = {zs_results[name][1]:.3f}")

### ★ Q1.2 Logit scale (4 pts)

The cell below rescales the ensemble logits to different values of $s$. It reports accuracy, mean confidence, and calibration error (ECE).

(a) Why does accuracy stay fixed while confidence changes?

(b) Which $s$ is best calibrated? Why might it differ from the $s$ learned in training? Recall that cosines lie in $[-1, 1]$.

(c) CLIP caps $s$ at 100. What would happen to the training loss and its gradients if $s$ were fixed at 1?

In [ ]:
def ece(probs, labels, n_bins=10):
    conf, pred = probs.max(1), probs.argmax(1)
    bins = np.linspace(0, 1, n_bins + 1)
    err = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            err += m.mean() * abs((pred[m] == labels[m]).mean() - conf[m].mean())
    return err

cos_ens = zs_results["8-template ensemble"][0] / LOGIT_SCALE
print(f"{'s':>6s} {'acc':>6s} {'conf':>6s} {'ECE':>6s}")
for s in [1, 10, 30, 50, LOGIT_SCALE, 300]:
    p = torch.softmax(s * cos_ens, dim=1).numpy()
    print(f"{s:6.0f} {(p.argmax(1) == cifar_labels).mean():6.3f} {p.max(1).mean():6.3f} {ece(p, cifar_labels):6.3f}")

**Your answer (Q1.2):**

*Write your answer here.*

### ★ Q1.3 Prompts (5 pts)

(a) Report the three accuracies above, and give one reason ensembling helps.

(b) Edit `my_templates`, report the accuracy you get, and explain your choice in 2–3 sentences. A thoughtful attempt gets full credit, even if it is lower.

In [ ]:
my_templates = ["a photo of a {}.", "a close-up photo of a {}."]   # <- edit me
W_mine = build_zero_shot_weights(CLASSES, my_templates, encode_text_b32)
acc_mine = (zero_shot_logits(cifar_img_emb, W_mine, LOGIT_SCALE).argmax(1).numpy() == cifar_labels).mean()
print(f"my templates: zero-shot accuracy = {acc_mine:.3f}")

**Your answer (Q1.3):**

*Write your answer here.*

### ★ Q1.4 Error analysis (5 pts)

(a) Name the two most-confused class pairs.

(b) Explain each pair. Consider the image content, the 32×32 resolution, and what web captions say about these classes.

(c) Name one limitation of zero-shot classification compared with a linear probe trained on CIFAR-10.

In [ ]:
pred = zs_results["8-template ensemble"][0].argmax(1).numpy()
cm = np.zeros((10, 10), dtype=int)
for y, p in zip(cifar_labels, pred):
    cm[y, p] += 1

fig, ax = plt.subplots(figsize=(6.2, 5.4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=60, ha="right")
ax.set_yticks(range(10)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
for i in range(10):
    for j in range(10):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8,
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.show()

off = cm + cm.T
np.fill_diagonal(off, 0)
pairs = sorted([(off[i, j], CLASSES[i], CLASSES[j]) for i in range(10) for j in range(i + 1, 10)], reverse=True)[:4]
print("most-confused pairs (both directions):", pairs)

wrong = np.where(pred != cifar_labels)[0][:8]
fig, axes = plt.subplots(1, len(wrong), figsize=(12, 1.9))
for ax, i in zip(axes, wrong):
    ax.imshow(cifar_images[i]); ax.axis("off")
    ax.set_title(f"{CLASSES[cifar_labels[i]]}\n→ {CLASSES[pred[i]]}", fontsize=8)
plt.show()

**Your answer (Q1.4):**

*Write your answer here.*

---
# Part 2: Rational Speech Acts (30 pts)

In a **reference game**, the speaker names a target among several objects, and the listener tries to pick it out. RSA uses a lexicon $[\![u]\!](o)\in\{0,1\}$, a prior $P(o)$, a rationality $\alpha>0$, and an utterance cost $C(u)$:

- $L_0(o \mid u) \propto [\![u]\!](o)\,P(o)$
- $S_1(u \mid o) \propto \exp\big(\alpha\,(\log L_0(o \mid u) - C(u))\big)$
- $L_1(o \mid u) \propto S_1(u \mid o)\,P(o)$

Shapes: `lexicon`, `L0` and `L1` are `[U, O]`, and `S1` is `[O, U]`. Every row is a distribution that sums to 1.

## 2.1 Implement RSA (15 pts, autograded)

Use NumPy only. `cost=None` means zero cost. Since $\log 0 = -\infty$, an utterance that is false of an object gets $S_1 = 0$. Use `np.errstate(divide="ignore")` and subtract the row max before exponentiating, so that you get no `nan`s and no warnings.

In [ ]:
# GRADED: rsa
import numpy as np


def literal_listener(lexicon, prior):
    """lexicon: [U, O] array of 0/1 (or nonnegative) truth values; prior: [O]. Returns L0: [U, O], rows sum to 1."""
    # YOUR CODE HERE
    raise NotImplementedError


def pragmatic_speaker(L0, alpha=1.0, cost=None):
    """L0: [U, O]; alpha > 0; cost: None or [U]. Returns S1: [O, U], rows sum to 1."""
    # YOUR CODE HERE
    raise NotImplementedError


def pragmatic_listener(S1, prior):
    """S1: [O, U]; prior: [O]. Returns L1: [U, O], rows sum to 1."""
    # YOUR CODE HERE
    raise NotImplementedError

### Frank & Goodman (2012)

The objects are a blue square, a blue circle and a green square. The utterances are *blue*, *green*, *square* and *circle*.

In [ ]:
FG_OBJECTS = ["blue square", "blue circle", "green square"]
FG_UTTS = ["blue", "green", "square", "circle"]
FG_LEXICON = np.array([[1, 1, 0],    # blue
                       [0, 0, 1],    # green
                       [1, 0, 1],    # square
                       [0, 1, 0]])   # circle
uniform = np.ones(3) / 3

L0_fg = literal_listener(FG_LEXICON, uniform)
S1_fg = pragmatic_speaker(L0_fg, alpha=1.0)
L1_fg = pragmatic_listener(S1_fg, uniform)
show = lambda M, rows, cols: display(pd.DataFrame(M, index=rows, columns=cols).round(3))
print("L0(o | u)"); show(L0_fg, FG_UTTS, FG_OBJECTS)
print("S1(u | o)"); show(S1_fg, FG_OBJECTS, FG_UTTS)
print("L1(o | u)"); show(L1_fg, FG_UTTS, FG_OBJECTS)
assert np.allclose(L1_fg[0], [0.6, 0.4, 0.0]), "L1('blue') should be [0.6, 0.4, 0] — check your implementation"
print("✓ L1('blue') = [0.6, 0.4, 0.0]")

### ★ Q2.1 Implicature (5 pts)

(a) Given *blue*, $L_0$ is indifferent between the two blue objects, but $L_1$ prefers the blue square. Explain why, using what the speaker *would have said* if the target were the blue circle.

(b) Which utterances does $L_1$ interpret exactly as $L_0$ does, and why?

**Your answer (Q2.1):**

*Write your answer here.*

### ★ Q2.2 Rationality α (5 pts)

(a) Describe the plot of $L_1(\text{blue square} \mid \textit{blue})$ against $\alpha$.

(b) What does $L_1$ become as $\alpha \to 0$, and as $\alpha \to \infty$? Justify both limits with the formula for $S_1$.

(c) Apply your speaker and listener once more to $L_1$, and report $L_2(\cdot \mid \textit{blue})$ at $\alpha = 1$. Does more recursion always sharpen the inference?

In [ ]:
alphas = np.logspace(-1.5, 1.2, 60)
vals = [pragmatic_listener(pragmatic_speaker(L0_fg, a), uniform)[0, 0] for a in alphas]
plt.figure(figsize=(5, 3.2))
plt.semilogx(alphas, vals, color="#2a78d6", lw=2)
plt.axhline(0.5, color="#898781", lw=1, ls="--")
plt.xlabel(r"speaker rationality $\alpha$"); plt.ylabel(r"$L_1$(blue square | 'blue')")
plt.ylim(0.45, 1.02); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# (c) one more level of recursion
# YOUR CODE HERE (not autograded: report what you find in the written answer below)

**Your answer (Q2.2):**

*Write your answer here.*

### ★ Q2.3 Cost and salience (5 pts)

The speaker may now also use the two-word utterances *blue square*, *blue circle* and *green square*, at cost $c$ each. The plot shows $S_1(\cdot \mid \text{blue square})$ against $c$, at $\alpha = 1$.

(a) At what $c$ is *blue square* as likely as *blue*? Derive the value, check it against the plot, and say in one sentence why $c$ acts like a per-word effort price.

(b) Go back to one-word utterances with no cost, and make the blue circle 3× as salient, with prior $\propto [1, 3, 1]$. Report $L_1(\cdot \mid \textit{blue})$, and explain how the prior competes with the implicature.

In [ ]:
FG2_UTTS = FG_UTTS + ["blue square", "blue circle", "green square"]
FG2_LEXICON = np.vstack([FG_LEXICON, np.eye(3)])
costs = np.linspace(0, 3, 61)
probs = np.array([pragmatic_speaker(literal_listener(FG2_LEXICON, uniform), 1.0,
                                     np.r_[np.zeros(4), np.full(3, c)])[0] for c in costs])
plt.figure(figsize=(5.2, 3.2))
assert np.allclose(probs[:, 0], probs[:, 2])          # 'blue' and 'square' are symmetric here
plt.plot(costs, probs[:, 0], lw=2, color="#2a78d6", label="'blue'  (= 'square')")
plt.plot(costs, probs[:, 4], lw=2, color="#1baf7a", label="'blue square'")
plt.xlabel("cost c of a two-word utterance"); plt.ylabel(r"$S_1$(u | blue square)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# (b) salient blue circle
# YOUR CODE HERE (not autograded: report what you find in the written answer below)

**Your answer (Q2.3):**

*Write your answer here.*

---
# Part 3: CLIP as the literal listener (35 pts)

We replace the hand-written lexicon with CLIP scores $s(i, u)$. Over the $C$ images in a context, the literal listener is

$$L_0(i \mid u) = \frac{\exp s(i,u)}{\sum_{i'} \exp s(i',u)},$$

which is exactly CLIP retrieval. RSA then builds the speaker and listener on top of it, with no training.

**The world:** 12 objects (4 colors × 3 shapes), each rendered 5 times, for 60 images.
- Part 3 uses 7 one-word utterances (the colors and the shapes), turned into prompts such as `"a red shape"` and `"a circle"`.
- Part 4 adds two-word utterances.

In [ ]:
COLORS = {"red": (220, 30, 30), "blue": (30, 70, 220), "green": (30, 160, 60), "yellow": (235, 200, 20)}
SHAPES = ["circle", "square", "triangle"]
OBJECTS = [(c, s) for c in COLORS for s in SHAPES]          # 12 object types


def render_shape(color, shape, seed, size=224):
    rng = np.random.RandomState(seed)
    img = Image.new("RGB", (size, size), (255, 255, 255))
    draw = ImageDraw.Draw(img)
    r = rng.randint(45, 70)
    cx, cy = rng.randint(r + 10, size - r - 10, 2)
    fill = COLORS[color]
    if shape == "circle":
        draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=fill)
    elif shape == "square":
        draw.rectangle([cx - r, cy - r, cx + r, cy + r], fill=fill)
    else:
        draw.polygon([(cx, cy - r), (cx - r, cy + r * 0.8), (cx + r, cy + r * 0.8)], fill=fill)
    return img


N_RENDERS = 5
shape_images, image_object = [], []
for k, (c, s) in enumerate(OBJECTS):
    for j in range(N_RENDERS):
        shape_images.append(render_shape(c, s, seed=100 * k + j))
        image_object.append(k)
image_object = np.array(image_object)

# vocabulary: 7 one-word utterances (Part 3) + 12 two-word utterances (Part 4)
UTTERANCES = list(COLORS) + SHAPES + [f"{c} {s}" for c, s in OBJECTS]
PROMPT = {u: (f"a {u} shape" if u in COLORS else f"a {u}") for u in UTTERANCES}
ONE_WORD = np.arange(7)                                      # indices of the 7 one-word utterances

fig, axes = plt.subplots(2, 12, figsize=(14, 2.6))
for k in range(12):
    for r in range(2):
        axes[r, k].imshow(shape_images[N_RENDERS * k + r]); axes[r, k].axis("off")
    axes[0, k].set_title(" ".join(OBJECTS[k]), fontsize=7)
plt.show()

Each listener is a score matrix `S[i, u]` of shape `[60 images, 19 utterances]`. There are two:

- **ViT-B/32** is the speaker's own model, and you compute its scores.
- **ViT-B/16** is a different CLIP that the speaker has never seen. Its scores are precomputed, which saves another 600 MB download.

In [ ]:
@torch.no_grad()
def score_matrix(model, processor, images, texts):
    img = F.normalize(encode_images(model, processor, images), dim=-1)
    txt = F.normalize(encode_texts(model, processor, texts), dim=-1)
    return (model.logit_scale.exp().item() * img @ txt.T).numpy()

prompts = [PROMPT[u] for u in UTTERANCES]
S_b32 = score_matrix(clip_b32, proc_b32, shape_images, prompts)


def load_scores(name):
    """Precomputed [60, 19] scores of another CLIP model on these same images and prompts."""
    path = f"data/scores_{name}.npy"
    if not os.path.exists(path):
        os.makedirs("data", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/zinengtang/cs183-fall26-a6/main/data/scores_{name}.npy", path)
    return np.load(path)


S_b16 = load_scores("b16")   # same as score_matrix(*load_clip("openai/clip-vit-base-patch16"), shape_images, prompts)

for name, S in [("ViT-B/32", S_b32), ("ViT-B/16", S_b16)]:
    color_acc = (S[:, :4].argmax(1) == image_object // 3).mean()
    shape_acc = (S[:, 4:7].argmax(1) == image_object % 3).mean()
    print(f"{name}: picks the right color word for {color_acc:.0%} of images, right shape word for {shape_acc:.0%}")

### Contexts

Each game has 3 images, and **image 0 is the target**.

- **random:** three different objects.
- **hard:** one distractor shares the target's color, and the other shares its shape (for example *red circle* vs. *red square* and *blue circle*). No single word is true of the target alone.

In [ ]:
def sample_contexts(kind, n, seed=0):
    rng = np.random.RandomState(seed)
    out = []
    while len(out) < n:
        types = rng.choice(len(OBJECTS), 3, replace=False)
        (tc, ts), d1, d2 = [OBJECTS[t] for t in types]
        if kind == "hard" and not ((d1[0] == tc and d2[1] == ts) or (d2[0] == tc and d1[1] == ts)):
            continue
        out.append(np.array([rng.choice(np.where(image_object == t)[0]) for t in types]))
    return out

CONTEXTS = {kind: sample_contexts(kind, 2000, seed=i) for i, kind in enumerate(["random", "hard"])}
print({k: len(v) for k, v in CONTEXTS.items()})

## 3.1 Neural RSA (15 pts, autograded)

Work in log space, because CLIP logits are around 20–35. `scores` is one context's `[C, U]` slice, with the target in row 0. Inputs and outputs are NumPy arrays. Use a numerically stable log-softmax.

- `context_log_L0(scores)` → `[U, C]`: a log-softmax over images.
- `literal_speaker(scores, target)` → `int`: the context-blind speaker $S_0 = \arg\max_u s(\text{target}, u)$.
- `neural_log_S1(log_L0, alpha, cost)` → `[C, U]`: a log-softmax over utterances of $\alpha(\log L_0 - C)$.
- `neural_log_L1(log_S1, log_prior)` → `[U, C]`: a log-softmax over images of $\log S_1 + \log P$, where `None` means a uniform prior.

In [ ]:
# GRADED: neural_rsa
import numpy as np


def _log_softmax(x, axis):
    x = x - x.max(axis=axis, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=axis, keepdims=True))


def context_log_L0(scores):
    """scores: [C, U] CLIP logits for the C context images. Returns log L0: [U, C]."""
    # YOUR CODE HERE
    raise NotImplementedError


def literal_speaker(scores, target=0):
    """Context-blind S0: index of the utterance with the highest score for the target image."""
    # YOUR CODE HERE
    raise NotImplementedError


def neural_log_S1(log_L0, alpha=1.0, cost=None):
    """log_L0: [U, C]. Returns log S1: [C, U]."""
    # YOUR CODE HERE
    raise NotImplementedError


def neural_log_L1(log_S1, log_prior=None):
    """log_S1: [C, U]; log_prior: None or [C]. Returns log L1: [U, C]."""
    # YOUR CODE HERE
    raise NotImplementedError

### Reference games

A speaker maps a context to an utterance. A listener maps a (context, utterance) pair to a guess. The game succeeds when the guess is image 0.

In [ ]:
def make_S0(S, utts=ONE_WORD):
    return lambda ctx: literal_speaker(S[ctx][:, utts], 0)

def make_S1(S, utts=ONE_WORD, alpha=1.0, cost=None):
    return lambda ctx: int(np.argmax(neural_log_S1(context_log_L0(S[ctx][:, utts]), alpha, cost)[0]))

def make_L0(S, utts=ONE_WORD):
    return lambda ctx, u: int(np.argmax(context_log_L0(S[ctx][:, utts])[u]))

def make_L1(S, utts=ONE_WORD, alpha=1.0):
    return lambda ctx, u: int(np.argmax(neural_log_L1(neural_log_S1(context_log_L0(S[ctx][:, utts]), alpha))[u]))

def play(speaker, listener, contexts):
    """Fraction of games in which the listener picks the target (index 0)."""
    return float(np.mean([listener(ctx, speaker(ctx)) == 0 for ctx in contexts]))

def show_game(ctx, utt_text, guesses=None, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(5.4, 2.1))
    for j, ax in enumerate(axes):
        ax.imshow(shape_images[ctx[j]]); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_edgecolor("#1baf7a" if j == 0 else "#c3c2b7"); sp.set_linewidth(3 if j == 0 else 1)
        if guesses and j in guesses:
            ax.set_xlabel(guesses[j], fontsize=8)
    axes[0].set_title("target", fontsize=8, color="#1baf7a")
    fig.suptitle(f"{title}  utterance: '{utt_text}'", fontsize=9); plt.tight_layout(); plt.show()

### Experiment A: context-blind vs. pragmatic speaker

The speaker and the listener are both ViT-B/32. This is self-play.

In [ ]:
L0_b32 = make_L0(S_b32)
expA = {}
for kind in ["random", "hard"]:
    for sname, spk in [("S0 (context-blind)", make_S0(S_b32)), ("S1 (pragmatic)", make_S1(S_b32))]:
        expA[(kind, sname)] = play(spk, L0_b32, CONTEXTS[kind])
display(pd.Series(expA).unstack().round(3).rename_axis(index="context", columns="speaker → L0 (B/32)"))

# one hard game where S0 fails and S1 succeeds
S0f, S1f = make_S0(S_b32), make_S1(S_b32)
for ctx in CONTEXTS["hard"]:
    u0, u1 = S0f(ctx), S1f(ctx)
    if L0_b32(ctx, u0) != 0 and L0_b32(ctx, u1) == 0:
        show_game(ctx, UTTERANCES[u0], {L0_b32(ctx, u0): "L0 guess"}, "S0:")
        show_game(ctx, UTTERANCES[u1], {L0_b32(ctx, u1): "L0 guess"}, "S1:")
        break

### ★ Q3.1 (6 pts)

(a) Report the four numbers.

(b) Using the example game, explain why $S_0$ fails in hard contexts.

(c) In hard contexts no single word is unique to the target, yet $S_1 \to L_0$ beats chance (1/3). What property of CLIP's *soft* lexicon does $S_1$ exploit? Look at `np.exp(context_log_L0(S_b32[ctx][:, ONE_WORD]))` for a few hard contexts.

**Your answer (Q3.1):**

*Write your answer here.*

### Experiment B: a new partner

The speaker still plans with ViT-B/32, but the actual listener is ViT-B/16. The oracle speaker plans with ViT-B/16 itself.

In [ ]:
L0_b16 = make_L0(S_b16)
expB = {}
for kind in ["random", "hard"]:
    expB[(kind, "S1(B/32) → L0(B/32)  [self-play]")] = play(make_S1(S_b32), L0_b32, CONTEXTS[kind])
    expB[(kind, "S1(B/32) → L0(B/16)  [new partner]")] = play(make_S1(S_b32), L0_b16, CONTEXTS[kind])
    expB[(kind, "S1(B/16) → L0(B/16)  [oracle]")] = play(make_S1(S_b16), L0_b16, CONTEXTS[kind])
display(pd.Series(expB).unstack(0).round(3))

# where do the two listeners disagree? average L0(target | u) for the utterance S1(B/32) chose, in hard contexts
agree = {"B/32": [], "B/16": []}
for ctx in CONTEXTS["hard"]:
    u = make_S1(S_b32)(ctx)
    agree["B/32"].append(np.exp(context_log_L0(S_b32[ctx][:, ONE_WORD])[u, 0]))
    agree["B/16"].append(np.exp(context_log_L0(S_b16[ctx][:, ONE_WORD])[u, 0]))
plt.figure(figsize=(4.2, 3.6))
plt.scatter(agree["B/32"], agree["B/16"], s=6, alpha=0.35, color="#2a78d6")
plt.plot([0, 1], [0, 1], color="#898781", lw=1, ls="--")
plt.xlabel("L0_B/32(target | u)  — what the speaker expects"); plt.ylabel("L0_B/16(target | u)  — what happens")
plt.title("hard contexts, u = S1(B/32)'s choice", fontsize=9); plt.tight_layout(); plt.show()

### ★ Q3.2 (7 pts)

(a) Report the drop from self-play to the new partner, in hard and in random contexts.

(b) Use the scatter plot to explain why hard contexts are fragile. What does the speaker exploit, and why might it not transfer to another model trained on the same data?

(c) Relate this to self-play conventions in multi-agent RL or emergent communication. Name one way a speaker could be more robust to an unknown listener.

**Your answer (Q3.2):**

*Write your answer here.*

### Experiment C: a pragmatic listener

The B/16 listener now runs one RSA step on its own model, giving $L_1$.

In [ ]:
expC = {}
for kind in ["random", "hard"]:
    for sname, spk in [("S0(B/32)", make_S0(S_b32)), ("S1(B/32)", make_S1(S_b32))]:
        for lname, lis in [("L0(B/16)", L0_b16), ("L1(B/16)", make_L1(S_b16))]:
            expC[(kind, sname, lname)] = play(spk, lis, CONTEXTS[kind])
display(pd.Series(expC).unstack().round(3))

# alpha sweep for the pragmatic listener in hard contexts
alphas_c = [0.1, 0.3, 1.0, 3.0, 10.0]
plt.figure(figsize=(4.6, 3.2))
for sname, spk, color in [("S0(B/32)", make_S0(S_b32), "#eb6834"), ("S1(B/32)", make_S1(S_b32), "#2a78d6")]:
    plt.semilogx(alphas_c, [play(spk, make_L1(S_b16, alpha=a), CONTEXTS["hard"]) for a in alphas_c],
                 "o-", color=color, label=f"{sname} → L1(B/16)")
    plt.axhline(play(spk, L0_b16, CONTEXTS["hard"]), color=color, ls=":", lw=1)
plt.xlabel(r"listener's assumed speaker rationality $\alpha$"); plt.ylabel("accuracy (hard)")
plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### ★ Q3.3 (7 pts)

(a) Report how accuracy changes from $L_0$ to $L_1$, for each speaker and each context type. The dotted lines in the plot are the $L_0$ baselines.

(b) Why does $L_1$ help so much in hard contexts? Relate your answer to Q2.1. Which speaker benefits more, and why?

(c) Can pragmatic listening hurt? When is $L_1$'s assumption about the speaker violated?

**Your answer (Q3.3):**

*Write your answer here.*

---
# Part 4 (283A only): Unknown partners (20 pts)

The speaker now plans against a **population** of two listeners: ViT-B/32 and LAION-2B ViT-B/32, which has the same architecture but different training data. It is tested on ViT-B/16, which stays held out. LAION's scores are precomputed, like B/16's.

We also add 12 two-word utterances, such as *red circle*, at cost $\lambda$ each. The speaker now trades informativity against brevity.

## 4.1 Population speaker (6 pts, autograded)

`population_speaker(log_L0_list, target, cost, mode)` returns the utterance $u$ that maximizes:

- for `"mean"`: $\frac{1}{K}\sum_k \log L_0^{(k)}(\text{target} \mid u) - C(u)$
- for `"min"`: $\min_k \log L_0^{(k)}(\text{target} \mid u) - C(u)$

Each `log_L0_list[k]` is `[U, C]`, and `cost` is `[U]`.

In [ ]:
# GRADED: population_speaker
import numpy as np


def population_speaker(log_L0_list, target, cost, mode="mean"):
    """Returns the index (int) of the utterance the population speaker chooses for `target`."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
S_laion = load_scores("laion")   # LAION-2B ViT-B/32, precomputed like B/16

ALL_UTTS = np.arange(len(UTTERANCES))
def cost_vec(lam):
    return np.r_[np.zeros(7), np.full(12, lam)]

def make_pop_speaker(score_list, lam, mode):
    return lambda ctx: population_speaker([context_log_L0(S[ctx]) for S in score_list], 0, cost_vec(lam), mode)

def make_S1_cost(S, lam):
    return make_S1(S, utts=ALL_UTTS, cost=cost_vec(lam))

L0_b16_all = make_L0(S_b16, ALL_UTTS)
LAM = 0.5
speakers_4 = {
    "S1(B/32)":              make_S1_cost(S_b32, LAM),
    "S1(LAION)":             make_S1_cost(S_laion, LAM),
    "pop-mean(B/32+LAION)":  make_pop_speaker([S_b32, S_laion], LAM, "mean"),
    "pop-min(B/32+LAION)":   make_pop_speaker([S_b32, S_laion], LAM, "min"),
    "S1(B/16) [oracle]":     make_S1_cost(S_b16, LAM),
}
rows = {}
for name, spk in speakers_4.items():
    utts = [spk(ctx) for ctx in CONTEXTS["hard"]]
    rows[name] = {"held-out L0(B/16) acc": play(spk, L0_b16_all, CONTEXTS["hard"]),
                  "% two-word": np.mean(np.array(utts) >= 7)}
display(pd.DataFrame(rows).T.round(3))

## ★ 4.2 Better, or just longer? (6 pts)

Sweep $\lambda$ over 400 hard contexts, for example with `[0, 0.1, 0.25, 0.5, 0.75, 1.0, 1.5]`. On one figure, plot held-out B/16 accuracy against the fraction of two-word utterances for `S1(B/32)`, `pop-mean` and `pop-min`. Label the axes and add a legend.

In [ ]:
# YOUR CODE HERE (not autograded: report what you find in the written answer below)

## ★ 4.3 Analysis (8 pts)

(a) Is the population speaker a *better* communicator, or just a more *verbose* one? Support your answer with numbers, comparing points of similar verbosity.

(b) Explain `pop-min` mechanistically. When does taking the worst case change the speaker's choice?

(c) Propose one method that should move the curve itself, and say how you would test it. Possible directions include calibrating the listeners, adapting from feedback, or a more diverse population. You do not need to implement it.

**Your answer (Q4.3):**

*Write your answer here.*

---
## Before you submit

1. *Runtime → Restart and run all.* 183 students only need Parts 1–3 to run, and may delete Part 4.
2. Export `HW6.pdf` and check that the plots rendered.
3. Upload `HW6.ipynb` and `HW6.pdf` to Gradescope **A6**, or the `submission.zip` built by `python3 make_submission.py`.